In [1]:
import numpy as np
import sympy as np
import matplotlib as plt

# Theme of the Project

This project investigates the mathematical foundations of cryptography through the implementation and analysis of classical and modern encryption schemes. Using Python and NumPy, we explore modular arithmetic and linear algebra as the theoretical basis for the Caesar cipher, Vigenère cipher, and the matrix-based Hill cipher, followed by a simplified demonstration of RSA encryption and cryptographic hash functions. Each notebook focuses on the underlying mathematics of one cipher, bridging the gap between abstract mathematical concepts and their practical application in secure communication.

## Mathematical Foundations

Before implementing any cipher, we establish the three mathematical 
tools that all cryptographic schemes in this project rely on:

- **Modular arithmetic** — the algebra of remainders, forming the 
  basis of all letter transformations
- **Greatest Common Divisor (GCD)** — used to determine whether a 
  number or matrix is invertible in modular space
- **Matrix operations (mod n)** — the foundation of block ciphers 
  such as the Hill cipher, where encryption is a linear transformation
  over a finite field

### Modular Arithmetic

Modular arithmetic is a mathematical system where integers "wrap around" 
after exceeding a given value called the **modulus**. Formally:

$$a \equiv b \pmod{m} \iff m \mid (a - b)$$

We say $a$ is congruent to $b$ modulo $m$ if $m$ divides their difference.

For example:

$$3 \equiv 15 \pmod{12} \quad \text{since } 12 \mid (15 - 3)$$
$$70 \equiv 10 \pmod{60} \quad \text{since } 60 \mid (70 - 10)$$

In cryptography, modular arithmetic keeps all values within a fixed range.
For classical ciphers this range is $\mathbb{Z}_{26} = \{0, 1, \dots, 25\}$,
where each integer represents a letter of the English alphabet:

$$27 \equiv 1 \pmod{26}$$
$$34 \equiv 8 \pmod{26}$$

Here it is in python code:

In [3]:
print(27 % 26)  # 1
print(34 % 26)  # 8

1
8


### Greatest Common Divisor (GCD)

The Greatest Common Divisor of two integers is the largest integer 
that divides both without a remainder:

$$\gcd(a, b) = \max\{d \in \mathbb{Z}^+ : d \mid a \text{ and } d \mid b\}$$

Two integers are called **coprime** if their GCD equals 1:

$$\gcd(a, b) = 1$$

This is not an edge case — it is the fundamental requirement in 
cryptography. A number $a$ has a modular inverse mod $m$ **if and 
only if** $\gcd(a, m) = 1$.

### The Euclidean Algorithm

Rather than factoring both numbers, we use the recursive observation:

$$\gcd(a, b) = \gcd(b,\ a \bmod b)$$

repeated until the remainder reaches zero. For example:

$$\gcd(48, 18) \rightarrow \gcd(18, 12) \rightarrow \gcd(12, 6) 
\rightarrow \gcd(6, 0) = 6$$

In the context of this project, the condition we require is:

$$\gcd(a, 26) = 1$$

For example $\gcd(3, 26) = 1$ so $3$ is a valid key, 
while $\gcd(13, 26) = 13$ so $13$ is not.

Here it is in python code it is pretty simple to implement:

In [9]:
def findGCD(a,b):
    if b == 0:
        return a
    return findGCD(b,a % b)    

In [13]:
print(findGCD(3,26))
print(findGCD(18,48))
print(findGCD(2,4))

1
6
2



### Modular Inverse

The modular inverse of $a$ modulo $m$ is an integer $x$ such that:

$$a \cdot x \equiv 1 \pmod{m}$$

For example, the modular inverse of $3$ mod $26$ is $9$, since:

$$3 \times 9 = 27 \equiv 1 \pmod{26}$$

A modular inverse exists **if and only if**:

$$\gcd(a, m) = 1$$

We find it using the **Extended Euclidean Algorithm**, which finds 
integers $x$ and $y$ satisfying Bézout's identity:

$$ax + my = \gcd(a, m)$$

When $\gcd(a, m) = 1$, the coefficient $x$ reduced mod $m$ is 
the modular inverse of $a$.

Here is this implemnted in python: 

In [29]:
def extented_euclidian(a,modular):
    curr_number = a
    next_number = modular
    current_coefficient = 1
    next_coefficient = 0
    while next_number != 0 : 
        q = curr_number // next_number
        curr_number , next_number = next_number, curr_number - next_number * q
        current_coefficient , next_coefficient = next_coefficient , current_coefficient - next_coefficient * q
    return curr_number,current_coefficient 
def mod_inverse(a,modular):
    gcd , x = extented_euclidian(a,modular)
    if gcd != 1:
        return None
    return x % modular         

In [30]:
print(mod_inverse(3 , 26))

9
